# LeetCode 2544 — Alternating Digit Sum bằng Python

Link đề: https://leetcode.com/problems/alternating-digit-sum/ · Độ khó: **Easy** · Chủ đề: Math, Digits, Simulation

Skeleton LeetCode:

```python
class Solution:
    def alternateDigitSum(self, n: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tính tổng các chữ số của `n` với **dấu xen kẽ**, chữ số **đầu tiên (hàng cao nhất)** mang dấu `+`, các chữ số sau lần lượt `-`, `+`, `-`, ...
2. Lấy chữ số bằng chuỗi `str(n)` (duyệt từ trái sang phải) và bằng `% 10`, `// 10` (duyệt từ phải sang trái).
3. Giải quyết khó khăn khi duyệt **từ phải sang trái**: dấu của chữ số cuối phụ thuộc vào **số chữ số** của `n` (chẵn/lẻ); và học một mẹo đẹp `acc = d - acc` không cần biết trước độ dài.
4. Viết 5 cách: chuỗi + chỉ số chẵn/lẻ, chuỗi + cắt lát `[::2]`, `%`/`//` với đảo dấu cuối, mẹo `acc = d - acc`, và đệ quy trên chuỗi `f(s) = s[0] - f(s[1:])`; đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: bắt đầu dấu `+` ở chữ số **cuối** thay vì **đầu**, đếm chỉ số từ 1 làm đảo dấu, bỏ sót chữ số cuối, gán dấu theo giá trị chữ số thay vì vị trí.

## 2. Đề bài & ví dụ

Cho số nguyên dương `n`. Mỗi chữ số của `n` được gán một dấu theo quy tắc:

- chữ số **đầu tiên (cao nhất)** mang dấu **dương**;
- mỗi chữ số còn lại mang dấu **ngược** với dấu của chữ số **đứng ngay trước nó** (nên dấu xen kẽ `+ - + - ...`).

Trả về **tổng** của tất cả chữ số kèm dấu.

| Ví dụ | n | Output | Giải thích |
|---|---|---|---|
| 1 | `521` | 4 | `(+5) + (-2) + (+1) = 4` |
| 2 | `111` | 1 | `(+1) + (-1) + (+1) = 1` |
| 3 | `886996` | 0 | `(+8) + (-8) + (+6) + (-9) + (+9) + (-6) = 0` |

Constraints: `1 <= n <= 10^9`.

In [1]:
n = 521
digits = [int(ch) for ch in str(n)]
signs = [1 if i % 2 == 0 else -1 for i in range(len(digits))]
print("chữ số:", digits)
print("dấu   :", signs)
print("tổng  :", sum(d * s for d, s in zip(digits, signs)))

chữ số: [5, 2, 1]
dấu   : [1, -1, 1]
tổng  : 4


## 3. Kiến thức nền

### 3.1. Duyệt chữ số bằng `str(n)` và `enumerate`

`str(521)` là `"521"`. `enumerate` cho cả **chỉ số** `i` (bắt đầu từ `0`) và ký tự. Chữ số ở chỉ số `0` là chữ số **cao nhất**. `int(ch)` đổi ký tự thành số.

In [2]:
for i, ch in enumerate(str(521)):
    print(f"chỉ số {i}: ký tự {ch!r} -> số {int(ch)}")

chỉ số 0: ký tự '5' -> số 5
chỉ số 1: ký tự '2' -> số 2
chỉ số 2: ký tự '1' -> số 1


### 3.2. Dấu xen kẽ theo chỉ số: `i % 2`, `(-1) ** i`

Chỉ số **chẵn** (`0, 2, 4, ...`) → dấu `+`; chỉ số **lẻ** → dấu `-`. Có thể viết `+1 if i % 2 == 0 else -1` hoặc `(-1) ** i` (`(-1)**0 = 1`, `(-1)**1 = -1`, ...). Chỉ số bắt đầu từ `0` là điều quan trọng: nếu đếm từ `1` thì dấu bị đảo (xem mục 13).

In [3]:
print([(-1) ** i for i in range(6)])
print([1 if i % 2 == 0 else -1 for i in range(6)])

[1, -1, 1, -1, 1, -1]
[1, -1, 1, -1, 1, -1]


### 3.3. Cắt lát với bước nhảy: `s[::2]` và `s[1::2]`

`s[::2]` lấy các phần tử ở chỉ số `0, 2, 4, ...`; `s[1::2]` lấy `1, 3, 5, ...`. Với `digits = [5, 2, 1]`: `digits[::2] = [5, 1]` (dấu `+`) và `digits[1::2] = [2]` (dấu `-`). Do đó tổng xen kẽ = `sum(digits[::2]) - sum(digits[1::2])`.

In [4]:
digits = [5, 2, 1]
print(digits[::2], digits[1::2])
print(sum(digits[::2]) - sum(digits[1::2]))

[5, 1] [2]
4


### 3.4. Tách chữ số từ **phải sang trái** bằng `%` và `//`

`n % 10` là chữ số cuối, `n // 10` bỏ chữ số cuối. Thứ tự lấy được là **ngược** (cuối trước, đầu sau). Khó khăn: dấu của chữ số **đầu** phải là `+`, nhưng ta chỉ biết nó ở **cuối** quá trình — nên dấu của chữ số cuối cùng của `n` phụ thuộc số chữ số của `n`.

In [5]:
n = 521
order = []
while n > 0:
    order.append(n % 10)
    n //= 10
print(order)                       # từ phải sang trái

[1, 2, 5]


### 3.5. Dấu của chữ số cuối phụ thuộc độ dài

Với `n` có `L` chữ số, chữ số ở **chỉ số** `i` (từ trái, tính từ 0) mang dấu `(-1)^i`. Chữ số **cuối** có chỉ số `L - 1` nên dấu là `(-1)^(L-1)`: `+` nếu `L` lẻ, `-` nếu `L` chẵn. Vì vậy nếu duyệt từ phải với dấu `+` ban đầu thì **kết quả đúng nếu `L` lẻ, ngược dấu nếu `L` chẵn**.

In [6]:
for n in [5, 52, 521, 5210]:
    L = len(str(n))
    sign_of_last = 1 if (L - 1) % 2 == 0 else -1
    print(f"n={n}: L={L}, dấu của chữ số cuối = {sign_of_last:+d}")

n=5: L=1, dấu của chữ số cuối = +1
n=52: L=2, dấu của chữ số cuối = -1
n=521: L=3, dấu của chữ số cuối = +1
n=5210: L=4, dấu của chữ số cuối = -1


### 3.6. Mẹo `acc = d - acc` (đảo dấu bằng phép trừ)

Gọi các chữ số của `n` từ trái sang phải là `a0 a1 a2 ... a(L-1)`. Tổng cần tìm là `a0 - a1 + a2 - ...`. Viết lồng dấu ngoặc: `a0 - (a1 - (a2 - (... - a(L-1))))`. Nhìn từ trong ra ngoài, ta bắt đầu với `acc = 0`, rồi lần lượt (từ chữ số **cuối** về chữ số **đầu**) làm `acc = a_i - acc`. Đây chính là vòng lặp lấy chữ số cuối bằng `% 10`. Vậy duyệt từ phải sang trái với `acc = d - acc` cho đúng kết quả **mà không cần biết độ dài**.

In [7]:
acc = 0
n = 521
while n > 0:
    d = n % 10
    acc = d - acc
    print(f"d={d}: acc = {d} - (acc cũ) = {acc}")
    n //= 10

d=1: acc = 1 - (acc cũ) = 1
d=2: acc = 2 - (acc cũ) = 1
d=5: acc = 5 - (acc cũ) = 4


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Cứ đọc số từ trái sang phải như đề: chữ số thứ nhất `+`, thứ hai `-`, thứ ba `+`, ...

Ví dụ `n = 886996` (6 chữ số):

```
chữ số : 8   8   6   9   9   6
dấu    : +   -   +   -   +   -
giá trị: +8  -8  +6  -9  +9  -6   -> tổng = 0
```

Dùng chuỗi `str(n)` rất tự nhiên vì ta lấy ngay chữ số đầu tiên ở chỉ số `0`. Cách làm bằng toán (`%`, `//`) đọc ngược từ phải sang trái nên cần xử lý dấu cẩn thận; đó là lý do bài này hay "bẫy" người mới.

## 5. Cách 1 — Brute Force / baseline

Đổi `n` thành chuỗi, duyệt từng ký tự với chỉ số `i` và cộng `+d` khi `i` chẵn, `-d` khi `i` lẻ. Chính là mô phỏng nguyên văn đề, và cũng là oracle của các test.

Nút thắt: không có đáng kể — `n <= 10^9` chỉ có tối đa 10 chữ số. Điều cần chú ý là chỉ số **bắt đầu từ 0** và **không được bỏ sót chữ số cuối**.

In [8]:
class SolutionString:
    def alternateDigitSum(self, n):
        total = 0
        for i, ch in enumerate(str(n)):
            if i % 2 == 0:
                total += int(ch)
            else:
                total -= int(ch)
        return total

print(SolutionString().alternateDigitSum(521))
print(SolutionString().alternateDigitSum(111))
print(SolutionString().alternateDigitSum(886996))

4
1
0


**Độ phức tạp:** Time `O(D)` với `D` là số chữ số của `n` (`D = O(log n)`), Space `O(D)` (chuỗi `str(n)`).

## 6. Ý tưởng thuật toán / bước đột phá

Có hai cách nhìn gọn:

1. **Tách theo vị trí chẵn/lẻ:** gom các chữ số ở chỉ số chẵn (dấu `+`) và lẻ (dấu `-`): `tổng chẵn - tổng lẻ`.
2. **Hệ thức đệ quy:** `f(s) = int(s[0]) - f(s[1:])` với `f('') = 0`: tổng xen kẽ của chuỗi chữ số bằng chữ số **đầu** trừ tổng xen kẽ của phần còn lại (dấu của phần còn lại đảo ngược, vì `a0 - a1 + a2 - ... = a0 - (a1 - a2 + ...)`). Hệ thức này khai triển thành `a0 - (a1 - (a2 - ...))`, và tính từ trong ra ngoài chính là vòng lặp `acc = d - acc` duyệt từ phải sang trái, **không cần biết độ dài**.

**Quy tắc "nếu… thì…":**

- Nếu duyệt từ **trái** → dấu của chữ số ở chỉ số `i` là `(-1)^i` (bắt đầu `+`).
- Nếu duyệt từ **phải** → dùng `acc = d - acc`; hoặc dùng dấu `+` rồi **đảo dấu cuối cùng nếu số chữ số chẵn**.
- Nếu `n` chỉ có một chữ số → đáp án chính là `n`.

Invariant (vòng lặp `acc = d - acc`): sau khi xử lý `k` chữ số cuối, `acc` bằng tổng xen kẽ của số tạo bởi `k` chữ số cuối đó, với chữ số **đầu tiên của đoạn đó** mang dấu `+`.

In [9]:
def alt(n):
    acc = 0
    while n > 0:
        acc = n % 10 - acc
        n //= 10
    return acc

print(alt(521), alt(111), alt(886996), alt(10), alt(7))

4 1 0 1 7


## 7. Các cách tiếp theo

Ta có 5 cách: (1) chuỗi + chỉ số chẵn/lẻ, (2) chuỗi + cắt lát `[::2]`, (3) `%`/`//` với đảo dấu cuối theo độ dài chẵn lẻ, (4) mẹo `acc = d - acc`, (5) đệ quy trên chuỗi `f(s) = int(s[0]) - f(s[1:])`.

### Cách 2 — Chuỗi + cắt lát

Biến thành danh sách chữ số, rồi `sum(digits[::2]) - sum(digits[1::2])`. Không cần `if` trong vòng lặp.

In [10]:
class SolutionSlice:
    def alternateDigitSum(self, n):
        digits = [int(ch) for ch in str(n)]
        return sum(digits[::2]) - sum(digits[1::2])

print(SolutionSlice().alternateDigitSum(521))
print(SolutionSlice().alternateDigitSum(111))
print(SolutionSlice().alternateDigitSum(886996))

4
1
0


Độ phức tạp Cách 2: Time `O(D)`, Space `O(D)`.

### Cách 3 — `%` và `//` với đảo dấu cuối

Duyệt từ phải sang trái, dấu bắt đầu là `+` và đổi mỗi bước. Khi kết thúc, **chữ số đầu** (xử lý sau cùng) có dấu `+` chỉ khi số chữ số `L` lẻ; nếu `L` chẵn thì cả kết quả bị ngược dấu → `return -total`. Đếm `L` ngay trong vòng lặp.

In [11]:
class SolutionRightSign:
    def alternateDigitSum(self, n):
        total = 0
        sign = 1
        length = 0
        while n > 0:
            total += sign * (n % 10)
            sign = -sign
            n //= 10
            length += 1
        return total if length % 2 == 1 else -total

print(SolutionRightSign().alternateDigitSum(521))
print(SolutionRightSign().alternateDigitSum(111))
print(SolutionRightSign().alternateDigitSum(886996))

4
1
0


Độ phức tạp Cách 3: Time `O(D)`, Space `O(1)`.

### Cách 4 — Mẹo `acc = d - acc`

Không cần dấu, không cần độ dài: mỗi bước `acc = (chữ số cuối) - acc`. Cách ngắn và không có lỗi dấu. Xem vì sao đúng ở mục 10.

In [12]:
class SolutionTrick:
    def alternateDigitSum(self, n):
        acc = 0
        while n > 0:
            acc = n % 10 - acc
            n //= 10
        return acc

print(SolutionTrick().alternateDigitSum(521))
print(SolutionTrick().alternateDigitSum(111))
print(SolutionTrick().alternateDigitSum(886996))

4
1
0


Độ phức tạp Cách 4: Time `O(D)`, Space `O(1)`.

### Cách 5 — Đệ quy

`f('') = 0`; ngược lại `f(s) = int(s[0]) - f(s[1:])`: chữ số đầu trừ tổng xen kẽ của phần còn lại. Độ sâu đệ quy chỉ `D <= 10`, nên an toàn. (Lưu ý: **không** viết `f(n) = n % 10 - f(n // 10)`; đó là tổng xen kẽ với chữ số **cuối** mang `+`, khác đề khi số chữ số chẵn.)

In [13]:
class SolutionRecursive:
    def alternateDigitSum(self, n):
        def f(s):
            if not s:
                return 0
            return int(s[0]) - f(s[1:])      # chữ số đầu trừ tổng xen kẽ của phần còn lại
        return f(str(n))

print(SolutionRecursive().alternateDigitSum(521))
print(SolutionRecursive().alternateDigitSum(111))
print(SolutionRecursive().alternateDigitSum(886996))

4
1
0


Độ phức tạp Cách 5: Time `O(D^2)` do cắt chuỗi `s[1:]` (với `D <= 10` không đáng kể), Space `O(D^2)` cho các chuỗi con trên ngăn xếp.

## 8. TRACE từng bước

Trace Cách 4 (`acc = d - acc`) với `n = 521` rồi `n = 10` (số chữ số chẵn, hay sai nếu dùng sai dấu):

| `n` còn lại | chữ số cuối `d` | `acc` mới = `d - acc` cũ |
|---|---|---|
| 521 | 1 | `1 - 0 = 1` |
| 52 | 2 | `2 - 1 = 1` |
| 5 | 5 | `5 - 1 = 4` |

Kết quả `4`. Với `n = 10`: `d = 0` → `acc = 0 - 0 = 0`; `d = 1` → `acc = 1 - 0 = 1`; kết quả `1` (`+1 - 0`).

In [14]:
def trace_acc(n):
    print("n =", n)
    acc = 0
    while n > 0:
        d = n % 10
        new = d - acc
        print(f"  n={n:>9} d={d} acc cũ={acc:>3} -> acc mới = {d} - ({acc}) = {new}")
        acc = new
        n //= 10
    print("  kết quả:", acc)
    return acc

trace_acc(521)
trace_acc(10)
trace_acc(886996)

n = 521
  n=      521 d=1 acc cũ=  0 -> acc mới = 1 - (0) = 1
  n=       52 d=2 acc cũ=  1 -> acc mới = 2 - (1) = 1
  n=        5 d=5 acc cũ=  1 -> acc mới = 5 - (1) = 4
  kết quả: 4
n = 10
  n=       10 d=0 acc cũ=  0 -> acc mới = 0 - (0) = 0
  n=        1 d=1 acc cũ=  0 -> acc mới = 1 - (0) = 1
  kết quả: 1
n = 886996
  n=   886996 d=6 acc cũ=  0 -> acc mới = 6 - (0) = 6
  n=    88699 d=9 acc cũ=  6 -> acc mới = 9 - (6) = 3
  n=     8869 d=9 acc cũ=  3 -> acc mới = 9 - (3) = 6
  n=      886 d=6 acc cũ=  6 -> acc mới = 6 - (6) = 0
  n=       88 d=8 acc cũ=  0 -> acc mới = 8 - (0) = 8
  n=        8 d=8 acc cũ=  8 -> acc mới = 8 - (8) = 0
  kết quả: 0


0

## 9. Code nộp LeetCode

In [15]:
class Solution:
    def alternateDigitSum(self, n: int) -> int:
        acc = 0
        while n > 0:
            acc = n % 10 - acc
            n //= 10
        return acc

print(Solution().alternateDigitSum(521), Solution().alternateDigitSum(111), Solution().alternateDigitSum(886996))

4 1 0


## 10. Vì sao thuật toán đúng

- **Cách 1, 2 (từ trái):** chữ số ở chỉ số `i` (tính từ chữ số cao nhất, bắt đầu 0) mang dấu `(-1)^i` — đúng định nghĩa: chữ số đầu `+`, mỗi chữ số sau ngược dấu chữ số đứng ngay trước. Cắt lát `[::2]` chính xác là các chỉ số chẵn, `[1::2]` là các chỉ số lẻ.
- **Cách 4 (`acc = d - acc`):** gọi các chữ số từ trái là `a0 a1 ... a(L-1)`. Vòng lặp xử lý `a(L-1)` trước, `a0` sau cùng, nên sau các bước ta có `acc = a0 - (a1 - (a2 - (... - a(L-1))))`. Khai triển dấu ngoặc: `a_i` bị đổi dấu đúng `i` lần (mỗi chữ số đứng trước nó thêm một dấu trừ), nên mang dấu `(-1)^i` — trùng định nghĩa của đề.
- **Cách 5 (đệ quy):** `f(s) = a0 - f(s[1:])` khai triển ra `a0 - (a1 - (a2 - ...))`, mỗi chữ số `a_i` bị đổi dấu `i` lần nên mang dấu `(-1)^i`; điều kiện dừng `f('') = 0`.
- **Cách 3:** nếu gán dấu `+` cho chữ số cuối (xử lý trước) và đổi dấu mỗi bước thì chữ số ở chỉ số `i` (từ trái) nhận dấu `(-1)^(L-1-i)`. Nhân với `(-1)^(L-1)` được `(-1)^i`: không đổi khi `L` lẻ, đảo toàn bộ khi `L` chẵn — nên `return -total` nếu `L` chẵn.
- **Kiểm chứng bằng máy:** test vét cạn `n = 1..20000` ở mục 12 đối chiếu tất cả cách với oracle.

## 11. Phân tích Time / Space Complexity

Gọi `D` = số chữ số của `n` (`D = floor(log10 n) + 1 <= 10`).

| Cách | Time | Space |
|---|---|---|
| 1. Chuỗi + chỉ số chẵn/lẻ | `O(D)` | `O(D)` |
| 2. Chuỗi + cắt lát | `O(D)` | `O(D)` |
| 3. `%`/`//` + đảo dấu cuối | `O(D)` | `O(1)` |
| 4. Mẹo `acc = d - acc` | `O(D)` | `O(1)` |
| 5. Đệ quy trên chuỗi | `O(D^2)` | `O(D^2)` |

Với `n <= 10^9` (`D <= 10`) mọi cách đều tức thì. Cách 4 có Space `O(1)` và không có nhánh dấu nên gọn nhất.

## 12. Bộ test

Test 1: ví dụ đề và các edge case (`n = 1`, `n = 10` — số chữ số chẵn, `n = 10^9` biên trên, số có chữ số `0`, số một chữ số).

In [16]:
impls = [SolutionString(), SolutionSlice(), SolutionRightSign(), SolutionTrick(), SolutionRecursive(), Solution()]
tests = [
    (521, 4),
    (111, 1),
    (886996, 0),
    (1, 1),                      # nhỏ nhất
    (9, 9),                      # một chữ số
    (10, 1),                     # 2 chữ số: +1 - 0 (bắt lỗi dấu từ phải)
    (12, -1),                    # +1 - 2
    (21, 1),                     # +2 - 1
    (100, 1),                    # +1 - 0 + 0
    (1000000000, 1),             # biên trên 10^9
    (999999999, 9),              # 9 chữ số 9: +9 -9 +9 -9 +9 -9 +9 -9 +9 = 9
    (123456789, 5),              # 1-2+3-4+5-6+7-8+9 = 5
    (98765432, 4),               # 9-8+7-6+5-4+3-2 = 4
]
def oracle(n):
    total, sign = 0, 1
    for ch in str(n):
        total += sign * int(ch)
        sign = -sign
    return total
for n, exp in tests:
    assert 1 <= n <= 10**9
    assert oracle(n) == exp, (n, oracle(n), exp)
    for im in impls:
        assert im.alternateDigitSum(n) == exp, (type(im).__name__, n, exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 13


Test 2: **vét cạn** `n = 1..20000` (bao phủ độ dài 1 đến 5 chữ số, chẵn lẫn lẻ), rồi ngẫu nhiên có seed trong toàn miền `1..10^9`.

In [17]:
import random
random.seed(2544)

for n in range(1, 20001):
    exp = oracle(n)
    for im in impls:
        assert im.alternateDigitSum(n) == exp, (type(im).__name__, n, exp)
print("vét cạn 1..20000 OK")

for _ in range(3000):
    n = random.choice([random.randint(1, 10**9), random.randint(10**8, 10**9), random.randint(1, 1000)])
    assert 1 <= n <= 10**9
    exp = oracle(n)
    for im in impls:
        assert im.alternateDigitSum(n) == exp, (type(im).__name__, n, exp)
print("3000 test ngẫu nhiên OK")

vét cạn 1..20000 OK


3000 test ngẫu nhiên OK


Test 3: tính chất — `|đáp án| <= 9 * ceil(D / 2)`; nếu `n` có một chữ số thì đáp án `= n`; nếu `n` có chữ số đầu `a` và phần còn lại `r` (`n = a * 10^(D-1) + r`) thì `f(n) = a - f'(r)` với `f'` tính trên chuỗi đủ `D - 1` chữ số (kể cả số `0` đứng đầu).

In [18]:
random.seed(44)
for _ in range(1000):
    n = random.randint(1, 10**9)
    s = str(n)
    ans = Solution().alternateDigitSum(n)
    assert abs(ans) <= 9 * ((len(s) + 1) // 2)
    if len(s) == 1:
        assert ans == n
    else:
        tail = s[1:]                         # giữ nguyên số 0 đứng đầu của phần còn lại
        tail_value = sum((-1) ** i * int(ch) for i, ch in enumerate(tail))
        assert ans == int(s[0]) - tail_value
print("tính chất (cận giá trị, một chữ số, hệ thức theo chữ số đầu): OK")

tính chất (cận giá trị, một chữ số, hệ thức theo chữ số đầu): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: duyệt từ phải sang trái với dấu `+` bắt đầu ở chữ số **cuối**

In [19]:
def wrong_from_right(n):
    total, sign = 0, 1
    while n > 0:
        total += sign * (n % 10)       # SAI: chữ số cuối nhận dấu +, nhưng dấu + phải thuộc chữ số ĐẦU
        sign = -sign
        n //= 10
    return total

n = 10
print("sai :", wrong_from_right(n))
print("đúng:", Solution().alternateDigitSum(n))
assert wrong_from_right(n) != Solution().alternateDigitSum(n)

sai : -1
đúng: 1


Vì sao sai: với `n = 10`, chữ số **đầu** là `1` (phải `+`), nhưng bản sai gán `+` cho chữ số cuối `0` và `-` cho `1` → `-1`. Kết quả chỉ đúng khi số chữ số **lẻ**. **Sửa:** đảo dấu nếu số chữ số chẵn, hoặc dùng `acc = d - acc`.

### Lỗi 2: đếm chỉ số từ 1 (dấu bị đảo)

In [20]:
def wrong_start_one(n):
    total = 0
    for i, ch in enumerate(str(n), start=1):      # SAI: i bắt đầu từ 1 nên chữ số đầu có i lẻ
        total += (-1) ** i * int(ch)
    return total

n = 521
print("sai :", wrong_start_one(n))
print("đúng:", Solution().alternateDigitSum(n))
assert wrong_start_one(n) != Solution().alternateDigitSum(n)

sai : -4
đúng: 4


Vì sao sai: `(-1) ** 1 = -1` nên chữ số đầu mang dấu `-`, toàn bộ kết quả bị **đảo dấu** (`-4` thay vì `4`). Chữ số đầu có chỉ số `0`. **Sửa:** `enumerate(...)` mặc định (từ 0).

### Lỗi 3: bỏ sót chữ số cuối (off-by-one)

In [21]:
def wrong_off_by_one(n):
    s = str(n)
    total = 0
    for i in range(len(s) - 1):                   # SAI: phải là range(len(s))
        total += (-1) ** i * int(s[i])
    return total

n = 521
print("sai :", wrong_off_by_one(n))
print("đúng:", Solution().alternateDigitSum(n))
assert wrong_off_by_one(n) != Solution().alternateDigitSum(n)

sai : 3
đúng: 4


Vì sao sai: `range(len(s) - 1)` dừng trước chữ số cuối, nên `521` chỉ tính `5 - 2 = 3` thay vì `5 - 2 + 1 = 4`. Số chữ số là `len(s)`, chỉ số hợp lệ `0..len(s)-1`.

### Lỗi 4: gán dấu theo **giá trị** chữ số thay vì **vị trí**

In [22]:
def wrong_by_value(n):
    return sum(d if d % 2 == 0 else -d for d in (int(ch) for ch in str(n)))   # SAI: chẵn/lẻ của giá trị chữ số

n = 521
print("sai :", wrong_by_value(n))
print("đúng:", Solution().alternateDigitSum(n))
assert wrong_by_value(n) != Solution().alternateDigitSum(n)

sai : -4
đúng: 4


Vì sao sai: dấu phụ thuộc **vị trí** (chỉ số chẵn/lẻ), không phụ thuộc chữ số đó chẵn hay lẻ. Với `521`, bản sai cho `-5 + 2 - 1 = -4`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng xen kẽ `+ - + -` theo vị trí" → duyệt chuỗi với `enumerate` (từ `0`) và `(-1) ** i`, hoặc cắt lát `[::2]` / `[1::2]`.
- Cần duyệt từ **phải sang trái** bằng `%`/`//` mà dấu tính từ **trái** → dùng `acc = d - acc` (không cần đếm độ dài).
- Hệ thức đệ quy: `f(s) = (chữ số đầu) - f(phần còn lại)` với `f('') = 0`. Cẩn thận: dùng `chữ số cuối - f(x // 10)` thì chữ số **cuối** mang `+`, sai khi số chữ số chẵn.
- Luôn thử `n = 10` (độ dài chẵn) và `n = 12` để bắt lỗi dấu; số có `0` ở giữa cũng nên thử.
- Chữ số đầu nằm ở chỉ số `0` của `str(n)`; chỉ số chẵn là dấu `+`.
- `n <= 10^9` chỉ ~10 chữ số nên đừng tối ưu hóa quá mức; ưu tiên cách rõ ràng và ít lỗi dấu.

## 15. Biến thể của bài

1. Chữ số **cuối** mang dấu `+` (đảo ngược quy ước) — khi đó cách duyệt từ phải sang trái với dấu `+` đúng.
2. Tính tổng xen kẽ của các số trong **mảng** (dấu theo chỉ số phần tử).
3. Tính tổng xen kẽ theo **cơ số khác** (ví dụ nhị phân hoặc hệ 16) — thay `10` bằng cơ số `b`.
4. Kiểm tra **chia hết cho 11**: một số chia hết cho 11 khi tổng xen kẽ các chữ số chia hết cho 11 (quy tắc chia hết).
5. Đếm số `n` trong `1..N` có tổng xen kẽ bằng `0` (DP chữ số).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Chuỗi + chỉ số chẵn/lẻ | dấu theo `i % 2` | `O(D)` | `O(D)` | **dễ hiểu nhất**, người mới |
| 2. Chuỗi + cắt lát | `sum([::2]) - sum([1::2])` | `O(D)` | `O(D)` | ngắn gọn, ít `if` |
| 3. `%`/`//` + đảo dấu cuối | dấu `+` từ phải, đảo nếu `D` chẵn | `O(D)` | `O(1)` | khi cần tránh chuỗi |
| 4. `acc = d - acc` | đảo dấu bằng phép trừ | `O(D)` | `O(1)` | gọn nhất, hay trong phỏng vấn |
| 5. Đệ quy trên chuỗi | `f(s) = s[0] - f(s[1:])` | `O(D^2)` | `O(D^2)` | thể hiện hệ thức toán học |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `alternateDigitSum(7340)` bằng bao nhiêu?

**Bài 2 (tính tay):** tính `alternateDigitSum(90210)` bằng cách ghi chữ số, dấu và tổng.

**Bài 3 (điền TODO):** hoàn thành `alt_sum_str(n)` dùng chuỗi và `enumerate` (từ 0).

**Bài 4 (tự viết lại):** viết `alt_sum_last_plus(n)` với quy ước **chữ số cuối** mang dấu `+` (đảo ngược đề), dùng `%` và `//` với dấu bắt đầu `+`.

**Bài 5 (mini challenge):** viết `divisible_by_11(n)` dùng `abs(alternateDigitSum(n)) % 11 == 0` và kiểm tra với `n` từ 1 đến 2000 so với `n % 11 == 0`.

In [23]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().alternateDigitSum(7340))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def alt_sum_str(n):
    total = 0
    # TODO: for i, ch in enumerate(str(n)): cộng +int(ch) nếu i chẵn, trừ nếu i lẻ
    return total

8


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** chữ số **đầu** mang `+`, rồi `-`, `+`, ... theo vị trí (không phụ thuộc giá trị chữ số).

**Mức 2 (vừa):** Bài 3: `if i % 2 == 0: total += int(ch) else: total -= int(ch)`. Bài 4: `total += sign * (n % 10)`, `sign = -sign`, `n //= 10`, không cần đảo dấu cuối. Bài 5: quy tắc chia hết cho 11 dựa trên tổng xen kẽ.

**Mức 3 (gần đáp án):** Bài 1: `7 - 3 + 4 - 0 = 8`. Bài 2: `9 - 0 + 2 - 1 + 0 = 10`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1: 7 - 3 + 4 - 0 = 8
assert Solution().alternateDigitSum(7340) == 8

# Bài 2: 9 - 0 + 2 - 1 + 0 = 10
assert Solution().alternateDigitSum(90210) == 10
assert [int(c) for c in "90210"] == [9, 0, 2, 1, 0]

# Bài 3
def alt_sum_str(n):
    total = 0
    for i, ch in enumerate(str(n)):
        if i % 2 == 0:
            total += int(ch)
        else:
            total -= int(ch)
    return total
assert alt_sum_str(521) == 4 and alt_sum_str(886996) == 0 and alt_sum_str(10) == 1

# Bài 4: chữ số cuối mang +
def alt_sum_last_plus(n):
    total, sign = 0, 1
    while n > 0:
        total += sign * (n % 10)
        sign = -sign
        n //= 10
    return total
assert alt_sum_last_plus(521) == 4          # 1 - 2 + 5 (đối xứng với số chữ số lẻ)
assert alt_sum_last_plus(10) == -1          # 0 - 1
assert alt_sum_last_plus(12) == 1           # 2 - 1

# Bài 5: quy tắc chia hết cho 11
def divisible_by_11(n):
    return abs(Solution().alternateDigitSum(n)) % 11 == 0
assert all(divisible_by_11(n) == (n % 11 == 0) for n in range(1, 2001))
assert divisible_by_11(121) and not divisible_by_11(122)
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "tổng xen kẽ các chữ số":**

```python
acc = 0
while n > 0:
    acc = n % 10 - acc      # chữ số cuối trừ đi tổng xen kẽ của phần còn lại
    n //= 10
return acc
```

(Hoặc từ trái: `sum(int(ch) * (-1) ** i for i, ch in enumerate(str(n)))`.)

**Checklist trước khi submit:**

- [ ] Chữ số **đầu** (cao nhất) mang dấu `+`, không phải chữ số cuối?
- [ ] Nếu dùng `enumerate`: chỉ số bắt đầu từ `0`?
- [ ] Không bỏ sót chữ số cuối (`range(len(s))`, không phải `len(s) - 1`)?
- [ ] Nếu duyệt từ phải bằng `%`: đã xử lý độ dài chẵn (hoặc dùng `acc = d - acc`)?
- [ ] Dấu theo **vị trí**, không theo chữ số chẵn/lẻ của giá trị?
- [ ] Đã thử `n = 10`, `n = 12`, `n = 10**9`, số có chữ số `0`?
- [ ] Chữ ký đúng `alternateDigitSum(self, n: int) -> int`?